In [3]:
!pip install -q transformers peft bitsandbytes accelerate
print("Done!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
Done!


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# import shutil, os

# # Copy from Drive back into Colab's fast local storage
# # We do this because reading from Drive during inference is slow
# # Colab's local disk is much faster for model loading

# base = "/content/drive/MyDrive/Week-8 Local/quantized"
# os.makedirs("./quantized/model-fp16", exist_ok=True)
# os.makedirs("./quantized/model-int4", exist_ok=True)

# shutil.copytree(f"{base}/model-fp16", "./quantized/model-fp16", dirs_exist_ok=True)
# shutil.copytree(f"{base}/model-int8", "./quantized/model-fp16", dirs_exist_ok=True)
# shutil.copytree(f"{base}/model-int4", "./quantized/model-int4", dirs_exist_ok=True)
# shutil.copy(f"{base}/model.gguf", "./quantized/model.gguf")

# print("Models copied locally!")
# print("fp16:", os.listdir("./quantized/model-fp16"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Models copied locally!
fp16: ['model.safetensors', 'tokenizer.model', 'tokenizer.json', 'config.json', 'tokenizer_config.json', 'generation_config.json', 'chat_template.jinja']


In [43]:
import torch
import time
import os
import csv
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TextStreamer
)

In [ ]:
# FP16_PATH = "./quantized/model-fp16"
# INT8_PATH  = "./quantized/model-int8"
# INT4_PATH = "./quantized/model-int4"
# GGUF_PATH = "./quantized/model.gguf"


DRIVE_BASE = "/content/drive/MyDrive/Week-8 Local"   

FP16_PATH  = f"{DRIVE_BASE}/quantized/model-fp16"
INT8_PATH  = f"{DRIVE_BASE}/quantized/model-int8"
INT4_PATH  = f"{DRIVE_BASE}/quantized/model-int4"
GGUF_PATH = f"{DRIVE_BASE}/quantized/model.gguf"

tokenizer = AutoTokenizer.from_pretrained(FP16_PATH)
tokenizer.pad_token = tokenizer.eos_token


test_prompts = [
    "### Instruction:\nWhat is machine learning?\n\n### Input:\n\n### Response:\n",
    "### Instruction:\nExplain neural networks in simple terms.\n\n### Input:\n\n### Response:\n",
    "### Instruction:\nWhat is the difference between AI and ML?\n\n### Input:\n\n### Response:\n",
]

os.makedirs("./benchmarks", exist_ok=True)
print("Setup done!")

Setup done!


In [ ]:
def benchmark(model, tokenizer, prompts, label):
    results = []

    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        
        torch.cuda.reset_peak_memory_stats()

        start = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
        elapsed = time.time() - start

        
        new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
        tok_per_sec = round(new_tokens / elapsed, 2)
        latency = round(elapsed, 3)
        vram = round(torch.cuda.max_memory_allocated() / 1e9, 3)

        response = tokenizer.decode(out[0], skip_special_tokens=True)
        response = response.split("### Response:")[-1].strip()[:150]

        print(f"[{label}] {tok_per_sec} tok/s | {latency}s | {vram}GB VRAM")
        print(f"Output: {response[:80]}\n")

        results.append({
            "format": label,
            "prompt": prompt.split("\n")[1][:40],
            "tokens_per_sec": tok_per_sec,
            "latency_sec": latency,
            "vram_gb": vram,
            "output": response,
            "accuracy": None
        })

    return results

In [ ]:

print("=== FP16 ===")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    FP16_PATH,
    dtype=torch.float16,
    device_map="auto"
)

results_fp16 = benchmark(model_fp16, tokenizer, test_prompts, "FP16")


del model_fp16
torch.cuda.empty_cache()
print("FP16 done!\n")

=== FP16 ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[FP16] 29.9 tok/s | 2.141s | 2.714GB VRAM
Output: Machine learning is a field of computer science and engineering that uses algori

[FP16] 27.1 tok/s | 3.69s | 2.716GB VRAM
Output: Neural networks are a type of artificial neural network that are designed to mim

[FP16] 34.57 tok/s | 2.487s | 2.715GB VRAM
Output: AI is an acronym for Artificial Intelligence, which is a branch of computer scie

FP16 done!



In [ ]:

print("=== INT8 ===")
model_int8 = AutoModelForCausalLM.from_pretrained(
    INT8_PATH,
    quantization_config=BitsAndBytesConfig(
        load_in_8bit=True,
    ),
    device_map="auto"
)

results_int8 = benchmark(model_int8, tokenizer, test_prompts, "INT8")

del model_int8
torch.cuda.empty_cache()
print("INT8 done!\n")

=== INT8 ===


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[INT8] 8.1 tok/s | 5.679s | 1.745GB VRAM
Output: Machine learning is a field of computer science and artificial intelligence that

[INT8] 9.74 tok/s | 8.317s | 1.747GB VRAM
Output: Neural networks are a type of artificial neural network that are designed to mim

[INT8] 9.02 tok/s | 11.081s | 1.748GB VRAM
Output: The main difference between AI and ML is that AI is a subset of ML, which is a b

INT8 done!



In [ ]:

print("=== INT4 ===")
model_int4 = AutoModelForCausalLM.from_pretrained(
    INT4_PATH,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16
    ),
    device_map="auto"
)

results_int4 = benchmark(model_int4, tokenizer, test_prompts, "INT4")

del model_int4
torch.cuda.empty_cache()
print("INT4 done!\n")

=== INT4 ===


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[INT4] 13.96 tok/s | 3.08s | 0.796GB VRAM
Output: Machine learning is a field of computer science that involves the development of

[INT4] 18.86 tok/s | 5.301s | 0.797GB VRAM
Output: Neural networks are a type of artificial neural network that are designed to mim

[INT4] 18.76 tok/s | 3.572s | 0.797GB VRAM
Output: The main difference between AI and ML is that AI is a subset of ML. AI is a set 

INT4 done!



In [18]:
!pip install -q llama-cpp-python
print("llama.cpp ready!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 MB 12.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.5 MB/s eta 0:00:00
llama.cpp ready!


In [49]:
from llama_cpp import Llama
import psutil, os

llm = Llama(model_path=GGUF_PATH, n_ctx=512, verbose=False)

def benchmark_gguf(llm, prompts):
    results = []
    for prompt in prompts:
        start = time.time()
        out = llm(prompt, max_tokens=100, echo=False)
        elapsed = round(time.time() - start, 3)
        response = out["choices"][0]["text"].strip()[:150]

        tokens_generated = out["usage"]["completion_tokens"]
        tok_per_sec = round(tokens_generated / elapsed, 2)

        print(f"[GGUF] {tok_per_sec} tok/s | {elapsed}s (CPU)")
        print(f"Output: {response[:80]}\n")

        results.append({
            "format": "GGUF",
            "prompt": prompt.split("\n")[1][:40],
            "tokens_per_sec": tok_per_sec,
            "latency_sec": elapsed,
            "vram_gb": f"{round(psutil.Process(os.getpid()).memory_info().rss / 1024**2, 1)}MB",
            "output": response,
            "accuracy": None
        })
    return results

print("=== GGUF (CPU) ===")
results_gguf = benchmark_gguf(llm, test_prompts)
print("GGUF done!")

llama_context: n_ctx_seq (512) < n_ctx_train (2048) -- the full capacity of the model will not be utilized


=== GGUF (CPU) ===
[GGUF] 5.89 tok/s | 11.04s (CPU)
Output: Machine learning is a field of artificial intelligence that involves using algor

[GGUF] 5.61 tok/s | 15.499s (CPU)
Output: Neural networks are a type of artificial neural system that consists of a sequen

[GGUF] 5.71 tok/s | 15.765s (CPU)
Output: AI is an acronym for artificial intelligence. It is a field of computer science 

GGUF done!


In [ ]:

print("=== STREAMING DEMO ===\n")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    FP16_PATH, torch_dtype=torch.float16, device_map="auto"
)

streamer = TextStreamer(tokenizer, skip_prompt=True)
inputs = tokenizer(test_prompts[0], return_tensors="pt").to("cuda")

with torch.no_grad():
    model_fp16.generate(**inputs, max_new_tokens=100, streamer=streamer, do_sample=False)

del model_fp16
torch.cuda.empty_cache()

=== STREAMING DEMO ===



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Machine learning is a field of computer science and engineering that uses algorithms and statistical techniques to learn from data. It is used to develop systems that can learn from experience and make decisions based on that experience. Machine learning is used in a variety of applications, including image recognition, natural language processing, and recommendation systems.</s>


In [ ]:

print("=== BATCH INFERENCE ===\n")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    FP16_PATH, torch_dtype=torch.float16, device_map="auto"
)


tokenizer.padding_side = "left"

batch_inputs = tokenizer(
    test_prompts, return_tensors="pt",
    padding=True, truncation=True, max_length=256
).to("cuda")

start = time.time()
with torch.no_grad():
    batch_out = model_fp16.generate(**batch_inputs, max_new_tokens=100, do_sample=False)
elapsed = round(time.time() - start, 2)

print(f"All 3 prompts in {elapsed}s\n")
for i, out in enumerate(batch_out):
    text = tokenizer.decode(out, skip_special_tokens=True)
    print(f"Prompt {i+1}: {text.split('### Response:')[-1].strip()[:100]}\n")

del model_fp16
torch.cuda.empty_cache()

=== BATCH INFERENCE ===



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

All 3 prompts in 3.23s

Prompt 1: Machine learning is a field of computer science and engineering that uses algorithms and statistical

Prompt 2: Neural networks are a type of artificial neural network that are designed to mimic the way the human

Prompt 3: AI is an acronym for Artificial Intelligence, which is a branch of computer science that deals with 



In [52]:
accuracy_checks = [
    "algorithm",
    "layer",
    "subset",
]

print("=== ACCURACY ===")
for label, results in [("FP16", results_fp16), ("INT8", results_int8), ("INT4", results_int4), ("GGUF", results_gguf)]:
    for i, r in enumerate(results):
        r["accuracy"] = 1 if accuracy_checks[i] in r["output"].lower() else 0
    correct = sum(r["accuracy"] for r in results)
    print(f"{label}: {correct}/3 ({round(correct/3*100)}%)")

=== ACCURACY ===
FP16: 2/3 (67%)
INT8: 2/3 (67%)
INT4: 3/3 (100%)
GGUF: 2/3 (67%)


In [53]:
all_results = results_fp16 + results_int8 + results_int4 + results_gguf

with open("./benchmarks/results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "format", "prompt", "tokens_per_sec", "latency_sec", "vram_gb", "output", "accuracy"
    ])
    writer.writeheader()
    writer.writerows(all_results)

print("Saved ./benchmarks/results.csv")

Saved ./benchmarks/results.csv


In [ ]:
print("\nFINAL BENCHMARK SUMMARY")
print(f"{'Format':<8} {'Avg Tok/s':<12} {'Avg Latency':<14} {'Memory':<16} {'Accuracy'}")
print("-" * 65)

for label, results in [("FP16", results_fp16), ("INT8", results_int8), ("INT4", results_int4), ("GGUF", results_gguf)]:
    avg_lat = round(sum(r["latency_sec"] for r in results) / len(results), 3)
    tok = results[0]["tokens_per_sec"]
    vram = results[0]["vram_gb"]
    correct = sum(r["accuracy"] for r in results)
    acc = f"{correct}/3 ({round(correct/3*100)}%)"

    
    mem = f"{vram} RAM" if "MB" in str(vram) else f"{vram}GB VRAM"

    print(f"{label:<8} {str(tok):<12} {avg_lat:<14} {mem:<16} {acc}")


FINAL BENCHMARK SUMMARY
Format   Avg Tok/s    Avg Latency    Memory           Accuracy
-----------------------------------------------------------------
FP16     29.9         2.773          2.714GB VRAM     2/3 (67%)
INT8     8.1          8.359          1.745GB VRAM     2/3 (67%)
INT4     13.96        3.984          0.796GB VRAM     3/3 (100%)
GGUF     5.89         14.101         2663.2MB RAM     2/3 (67%)


In [ ]:
avg_fp16 = round(sum(r["latency_sec"] for r in results_fp16) / len(results_fp16), 3)
avg_int8 = round(sum(r["latency_sec"] for r in results_int8) / len(results_int8), 3)   
avg_int4 = round(sum(r["latency_sec"] for r in results_int4) / len(results_int4), 3)
avg_gguf = round(sum(r["latency_sec"] for r in results_gguf) / len(results_gguf), 3)

report = f"""# Benchmark Report — Day 4

## Models Tested
- FP16 : full precision fine-tuned model (GPU)
- INT8 : 8-bit quantised model (GPU)
- INT4 : 4-bit quantised model (GPU)
- GGUF : q8_0 quantised model (CPU via llama.cpp)

## Results

| Format | Tokens/sec | Avg Latency | Memory | Accuracy |
|--------|------------|-------------|--------|----------|
| FP16   | {results_fp16[0]["tokens_per_sec"]} | {avg_fp16}s | {results_fp16[0]["vram_gb"]}GB VRAM | {sum(r["accuracy"] for r in results_fp16)}/3 |
| INT8   | {results_int8[0]["tokens_per_sec"]} | {avg_int8}s | {results_int8[0]["vram_gb"]}GB VRAM | {sum(r["accuracy"] for r in results_int8)}/3 |
| INT4   | {results_int4[0]["tokens_per_sec"]} | {avg_int4}s | {results_int4[0]["vram_gb"]}GB VRAM | {sum(r["accuracy"] for r in results_int4)}/3 |
| GGUF   | {results_gguf[0]["tokens_per_sec"]} | {avg_gguf}s | {results_gguf[0]["vram_gb"]} RAM | {sum(r["accuracy"] for r in results_gguf)}/3 |

## Features Demonstrated
- Streaming output — tokens appear live as generated
- Batch inference — 3 prompts processed simultaneously
- Multi-prompt testing — 3 different prompts per format

## Key Findings
- INT8 uses less VRAM than FP16 with near-identical output quality
- INT4 uses the least VRAM of the GPU formats
- GGUF runs with zero GPU — works on any laptop
- Batch inference handles multiple users faster than one by one
"""

with open("BENCHMARK-REPORT.md", "w") as f:
    f.write(report)

print(report)

# Benchmark Report — Day 4

## Models Tested
- FP16 : full precision fine-tuned model (GPU)
- INT8 : 8-bit quantised model (GPU)
- INT4 : 4-bit quantised model (GPU)
- GGUF : q8_0 quantised model (CPU via llama.cpp)

## Results

| Format | Tokens/sec | Avg Latency | Memory | Accuracy |
|--------|------------|-------------|--------|----------|
| FP16   | 29.9 | 2.773s | 2.714GB VRAM | 2/3 |
| INT8   | 8.1 | 8.359s | 1.745GB VRAM | 2/3 |
| INT4   | 13.96 | 3.984s | 0.796GB VRAM | 3/3 |
| GGUF   | 5.89 | 14.101s | 2663.2MB RAM | 2/3 |

## Features Demonstrated
- Streaming output — tokens appear live as generated
- Batch inference — 3 prompts processed simultaneously
- Multi-prompt testing — 3 different prompts per format

## Key Findings
- INT8 uses less VRAM than FP16 with near-identical output quality
- INT4 uses the least VRAM of the GPU formats
- GGUF runs with zero GPU — works on any laptop
- Batch inference handles multiple users faster than one by one



In [57]:
!cp /content/BENCHMARK-REPORT.md /content/drive/MyDrive/Week-8\ Local